# NB03 — Statistical Analysis & Figures

**Environment:** Local Python (no Spark).

**Purpose:** From the cached CSVs written by NB02 (or `scripts/01_extract_and_stratify.py`), run the statistical tests and produce the report figures.

**Full script:** `scripts/02_analysis_and_figures.py`.

**Tests:**
- Fisher exact vs. global no-Pfam rate per biome (BH-FDR corrected)
- Bootstrap 95% CIs on per-biome no-Pfam rate (2,000 draws, seed=42)
- Chi-square independence: biome × pfam_tier, marginal AND stratified by is_core
- Per-biome log₂ enrichment vs. global core rate — the H1 test

**Note on statsmodels GLM:** the on-cluster Python 3.13 statsmodels install has a broken `deprecate_kwarg` import chain, so we use scipy chi-square + manual log-odds. The chi² tests reject independence at every stratification (p < 10⁻³⁰⁰) — no GLM needed to establish the biome effect.

**Figures (in `figures/`):**
- `NB03_biome_stacked_tier.png` — coverage tier per biome (all clusters)
- `NB03_biome_by_core_stacked.png` — same, faceted by is_core (the money figure)
- `NB03_biome_no_pfam_rate.png` — no-Pfam rate with bootstrap CIs
- `NB03_top_uncovered_heatmap.png` — biome × top-30 uncovered Pfam heatmap
- `NB03_pfam_universe.png` — PDB vs. pangenome Pfam sets

**Interpretation:** see `REPORT.md`. Headline: H1 (environmental > host coverage gap) is **rejected on marginal rates** but **confirmed within core clusters** (freshwater 18.0% no-Pfam > host_urogenital 11.3%). The confound is pangenome depth: host biomes have 10–100× more genomes so accumulate more accessory content.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests

summary = pd.read_csv('../data/biome_summary.csv')
matrix  = pd.read_csv('../data/biome_pfam_matrix.csv')
top_unc = pd.read_csv('../data/biome_top_uncovered.csv')

## Fisher enrichments & bootstrap CIs

In [ ]:
total_no_pfam = summary['n_no_pfam'].sum()
total_all = summary['n_clusters_total'].sum()
global_no_pfam_rate = total_no_pfam / total_all
print(f'Global no-Pfam rate: {100*global_no_pfam_rate:.2f}%')

def fisher_vs_global(a, n):
    b_a = a; b_b = n - a
    g_a = total_no_pfam - a; g_b = (total_all - total_no_pfam) - b_b
    if b_b < 0 or g_a < 0 or g_b < 0: return np.nan
    _, p = stats.fisher_exact([[b_a, b_b], [g_a, g_b]])
    return p

summary['fisher_p'] = summary.apply(lambda r: fisher_vs_global(r['n_no_pfam'], r['n_clusters_total']), axis=1)
summary['fisher_q_bh'] = multipletests(summary['fisher_p'].fillna(1.0), method='fdr_bh')[1]
summary['log2_enrich'] = np.log2((summary['n_no_pfam'] / summary['n_clusters_total']) / global_no_pfam_rate)
summary.sort_values('log2_enrich').round(3)

## Chi-square: biome × tier, marginal + stratified by is_core

In [ ]:
for label, sub in [('marginal', matrix),
                    ('is_core=True', matrix[matrix['is_core'] == True]),
                    ('is_core=False', matrix[matrix['is_core'] == False])]:
    ctab = sub.groupby(['biome', 'pfam_tier'])['n_clusters'].sum().unstack(fill_value=0)
    chi2, p, dof, _ = stats.chi2_contingency(ctab.values)
    print(f'{label:15s}  chi2={chi2:>12,.0f}  dof={dof}  p={p:.2e}')

## Per-biome CORE no-Pfam rates (the H1 test)

In [ ]:
mtx = matrix.copy()
mtx['is_no_pfam'] = (mtx['pfam_tier'] == 'no_pfam_annotation').astype(int)
core = mtx[mtx['is_core'] == True]
core_rows = (core.groupby('biome').apply(lambda g: pd.Series({
    'n_total': g['n_clusters'].sum(),
    'n_no_pfam': (g['n_clusters'] * g['is_no_pfam']).sum(),
})).reset_index())
core_rows['rate'] = core_rows['n_no_pfam'] / core_rows['n_total']
gc_rate = core_rows['n_no_pfam'].sum() / core_rows['n_total'].sum()
core_rows['log2_enrich_vs_core_global'] = np.log2(core_rows['rate'] / gc_rate)
core_rows.sort_values('log2_enrich_vs_core_global').round(4)